Import the libraries.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression

Read the data

In [ ]:
df = np.round(pd.read_csv('Datasets/50_Startups.csv')[['R&D Spend','Administration','Marketing Spend','Profit']]/10000)
np.random.seed(9)
df = df.sample(5)
df

,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


In [ ]:
df = df.iloc[:,0:-1]
df


,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


In [ ]:
df.iloc[1,0] = np.nan
df.iloc[3,1] = np.nan
df.iloc[-1,-1] = np.nan
df.head()

/tmp/ipykernel_9449/1429726703.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[1,0] = np.nan
/tmp/ipykernel_9449/1429726703.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[3,1] = np.nan
/tmp/ipykernel_9449/1429726703.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[-1,-1] = np.nan


,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


Make another dataframe and fill the null values with the mean of the column.

In [ ]:
df0 = pd.DataFrame()
df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())
df0


,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,9.25,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


Copy it to new dataframe and make one location of one column as null (as in original dataframe).

In [ ]:
df1 = df0.copy()
df1.iloc[1,0] = np.nan
df1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.25,26.00
44,2.0,15.00,29.25


Train the model and predict the value of that null cell.

In [ ]:
x = df1.iloc[[0,2,3,4], 1:3]
y = df1.iloc[[0,2,3,4], 0]
model = LinearRegression()
model.fit(x,y)
df1.iloc[1,0] = model.predict(df1.iloc[1,1:3].values.reshape(1,2))
df1


/home/hardik/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


,R&D Spend,Administration,Marketing Spend
21,8.000000,15.00,30.00
37,23.141587,5.00,20.00
2,15.000000,10.00,41.00
14,12.000000,11.25,26.00
44,2.000000,15.00,29.25


Repeat the same steps for other 2 column also and get the predictedd values.

In [ ]:
df1.iloc[3,1] = np.nan
x = df1.iloc[[0,1,2,4], [0,2]]
y = df1.iloc[[0,1,2,4], 1]
model.fit(x,y)
df1.iloc[3,1] = model.predict(df1.iloc[3,[0,2]].values.reshape(1,2))
df1

/home/hardik/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


,R&D Spend,Administration,Marketing Spend
21,8.000000,15.000000,30.00
37,23.141587,5.000000,20.00
2,15.000000,10.000000,41.00
14,12.000000,11.063618,26.00
44,2.000000,15.000000,29.25


In [ ]:
df1.iloc[4,-1] = np.nan
x = df1.iloc[0:4,0:2]
y = df1.iloc[0:4,-1]
model.fit(x,y)
df1.iloc[4,-1] = model.predict(df1.iloc[4,0:2].values.reshape(1,2))
df1


/home/hardik/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


,R&D Spend,Administration,Marketing Spend
21,8.000000,15.000000,30.000000
37,23.141587,5.000000,20.000000
2,15.000000,10.000000,41.000000
14,12.000000,11.063618,26.000000
44,2.000000,15.000000,31.601847


Above same steps can be done using imputive iterator function.

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression

# Copy your dataframe
dfnew = df.copy()

# Introduce missing values
dfnew.iloc[1, 0] = np.nan
dfnew.iloc[3, 1] = np.nan
dfnew.iloc[4, -1] = np.nan

# Define imputer
imputer = IterativeImputer(
    estimator=LinearRegression(),
    max_iter=1, # IN this case 1 to mimic the loop.
    random_state=0
)

# Fit and transform
df_imputed = imputer.fit_transform(dfnew)

# Convert back to DataFrame (important)
df_imputed = pd.DataFrame(df_imputed, columns=dfnew.columns)

print(df_imputed)

   R&D Spend  Administration  Marketing Spend
0   8.000000       15.000000        30.000000
1  23.141587        5.000000        20.000000
2  15.000000       10.000000        41.000000
3  12.000000       11.063618        26.000000
4   2.000000       15.000000        31.601847


/home/hardik/.local/lib/python3.10/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Get the difference btwn original and prdicted values.

In [ ]:
df1-df0

,R&D Spend,Administration,Marketing Spend
21,0.000000,0.000000,0.000000
37,13.891587,0.000000,0.000000
2,0.000000,0.000000,0.000000
14,0.000000,-0.186382,0.000000
44,0.000000,0.000000,2.351847


Repeat the whole process till the difference get 0 or close to it. The close to 0 it is, the more accurate it is.

In [ ]:
df2 = df1.copy()
df2.iloc[1,0] = np.nan
x = df2.iloc[[0,2,3,4], 1:3]
y = df2.iloc[[0,2,3,4], 0]
model.fit(x,y)
df2.iloc[1,0] = model.predict(df2.iloc[1,1:3].values.reshape(1,2))

df2.iloc[3,1] = np.nan
x = df2.iloc[[0,1,2,4], [0,2]]
y = df2.iloc[[0,1,2,4], 1]
model.fit(x,y)
df2.iloc[3,1] = model.predict(df2.iloc[3,[0,2]].values.reshape(1,2))
    
df2.iloc[4,-1] = np.nan
x = df2.iloc[0:4,0:2]
y = df2.iloc[0:4,-1]
model.fit(x,y)
df2.iloc[4,-1] = model.predict(df2.iloc[4,0:2].values.reshape(1,2))
df2

/home/hardik/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/home/hardik/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/home/hardik/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


,R&D Spend,Administration,Marketing Spend
21,8.000000,15.000000,30.000000
37,23.828703,5.000000,20.000000
2,15.000000,10.000000,41.000000
14,12.000000,11.230737,26.000000
44,2.000000,15.000000,39.390436


In [ ]:
df2-df1

,R&D Spend,Administration,Marketing Spend
21,0.000000,0.000000,0.000000
37,0.687117,0.000000,0.000000
2,0.000000,0.000000,0.000000
14,0.000000,0.167119,0.000000
44,0.000000,0.000000,7.788589


In [ ]:
df3 = df2.copy()

df3.iloc[1,0] = np.nan
x = df3.iloc[[0,2,3,4], 1:3]
y = df3.iloc[[0,2,3,4], 0]
model.fit(x,y)
df3.iloc[1,0] = model.predict(df3.iloc[1,1:3].values.reshape(1,2))

df3.iloc[3,1] = np.nan
x = df3.iloc[[0,1,2,4], [0,2]]
y = df3.iloc[[0,1,2,4], 1]
model.fit(x,y)
df3.iloc[3,1] = model.predict(df3.iloc[3,[0,2]].values.reshape(1,2))

df3.iloc[4,-1] = np.nan
x = df3.iloc[0:4,0:2]
y = df3.iloc[0:4,-1]
model.fit(x,y)
df3.iloc[4,-1] = model.predict(df3.iloc[4,0:2].values.reshape(1,2))

df3

/home/hardik/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/home/hardik/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/home/hardik/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


,R&D Spend,Administration,Marketing Spend
21,8.000000,15.000000,30.000000
37,26.887784,5.000000,20.000000
2,15.000000,10.000000,41.000000
14,12.000000,12.275798,26.000000
44,2.000000,15.000000,63.339357


In [ ]:
df3-df2

,R&D Spend,Administration,Marketing Spend
21,0.00000,0.00000,0.00000
37,3.05908,0.00000,0.00000
2,0.00000,0.00000,0.00000
14,0.00000,1.04506,0.00000
44,0.00000,0.00000,23.94892
